# Can we predict 30-day readmission?
Data: `data.csv` (one row per hospital stay), dictionary in `columns.csv`. Target: `readmit_30d`.

Plan: inspect data, set baselines, fit models with **person-grouped** cross-validation, then judge whether the signal is real and useful.

Key design point: the 11.9k rows are stays from only ~1,500 people (~8 stays each). Rows from the same person are not independent, so a random row split would put the same person in train and test and inflate scores. All evaluation keeps each person entirely in one fold. `person_key` is never used as a feature.

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
from sklearn.calibration import calibration_curve
from sklearn.inspection import permutation_importance
SEED = 0
df = pd.read_csv('data.csv')
print(pd.read_csv('columns.csv').to_string(index=False))
df.head()

## 1. Data checks

In [ ]:
print(df.shape, '| people:', df.person_key.nunique(), '| duplicate rows:', df.duplicated().sum(), '| missing cells:', df.isna().sum().sum())
p = df.readmit_30d.mean()
print(f'Readmission rate: {p:.3f}')
df.describe().T

In [ ]:
# Is the outcome clustered within person? (matters for how we split)
g = df.groupby('person_key').readmit_30d.agg(['mean', 'size'])
print(f"Between-person SD of readmit rate: observed {g['mean'].std():.3f}")
print(f"Expected if stays were independent: {np.sqrt(p*(1-p)/g['size']).mean():.3f}")
print(df.groupby('visit_number').readmit_30d.mean().round(3).head(10).to_dict())

## 2. Each feature vs. readmission (rate by decile)

In [ ]:
feats = ['visit_number', 'birth_year', 'bmi', 'length_of_stay_days', 'n_medications', 'lab_a', 'lab_b', 'postcode_area']
fig, ax = plt.subplots(2, 4, figsize=(16, 7))
for a, f in zip(ax.ravel(), feats):
    q = pd.qcut(df[f], 10, duplicates='drop')
    r = df.groupby(q, observed=True).readmit_30d.mean()
    a.plot(range(len(r)), r.values, 'o-'); a.axhline(p, color='grey', ls='--')
    a.set_title(f); a.set_xlabel('decile'); a.set_ylim(0, .5)
plt.tight_layout(); plt.show()
print(df.groupby('sex').readmit_30d.mean())
print(df[feats + ['readmit_30d']].corr()['readmit_30d'].drop('readmit_30d').round(3))

## 3. Models with grouped cross-validation

In [ ]:
X = df.drop(columns=['person_key', 'readmit_30d']); y = df.readmit_30d; groups = df.person_key
num = ['visit_number', 'birth_year', 'bmi', 'length_of_stay_days', 'n_medications', 'lab_a', 'lab_b']
cat = ['sex', 'postcode_area']   # postcode is a code, not a quantity -> categorical
pre = ColumnTransformer([('n', StandardScaler(), num), ('c', OneHotEncoder(handle_unknown='ignore'), cat)])
models = {
    'Prior only (dummy)': DummyClassifier(strategy='prior'),
    'Logistic regression': make_pipeline(pre, LogisticRegression(max_iter=2000)),
    'Gradient boosting': HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=200,
                                                        l2_regularization=1.0, random_state=SEED),
}
Xgb = X.assign(sex=(X.sex == 'M').astype(int))   # trees get postcode as a plain number
cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
folds = list(cv.split(X, y, groups))
oof = {}; rows = []
for n, m in models.items():
    XX = Xgb if n == 'Gradient boosting' else X
    pr = np.zeros(len(df)); aucs = []
    for tr, te in folds:
        m.fit(XX.iloc[tr], y.iloc[tr]); pr[te] = m.predict_proba(XX.iloc[te])[:, 1]
        aucs.append(roc_auc_score(y.iloc[te], pr[te]))
    oof[n] = pr
    rows.append(dict(model=n, AUC=roc_auc_score(y, pr), AUC_fold_sd=np.std(aucs),
                     PR_AUC=average_precision_score(y, pr), Brier=brier_score_loss(y, pr)))
res = pd.DataFrame(rows).set_index('model').round(4)
print(f'Base rate (= PR-AUC of a useless model): {p:.3f}')
res

Uncertainty on AUC: bootstrap over **people**, not rows.

In [ ]:
rng = np.random.default_rng(SEED); pids = df.person_key.unique(); idx = df.groupby('person_key').indices
def boot(pr, B=300):
    out = []
    for _ in range(B):
        s = rng.choice(pids, len(pids)); ii = np.concatenate([idx[k] for k in s])
        out.append(roc_auc_score(y.iloc[ii], pr[ii]))
    return np.percentile(out, [2.5, 97.5])
for n in ['Logistic regression', 'Gradient boosting']:
    lo, hi = boot(oof[n]); print(f'{n}: AUC 95% CI {lo:.3f}-{hi:.3f}')

Which features carry the signal? Permutation importance on held-out folds (drop in AUC).

In [ ]:
lr = models['Logistic regression']; imp = np.zeros(X.shape[1])
for tr, te in folds:
    lr.fit(X.iloc[tr], y.iloc[tr])
    r = permutation_importance(lr, X.iloc[te], y.iloc[te], scoring='roc_auc', n_repeats=10, random_state=SEED)
    imp += r.importances_mean / len(folds)
pd.Series(imp, index=X.columns).sort_values().plot.barh(title='Held-out permutation importance, logistic')
plt.show()

## 4. Discrimination, calibration and usefulness (out-of-fold predictions)

In [ ]:
best = res.drop('Prior only (dummy)').AUC.idxmax(); pr = oof[best]; print('Best model:', best)
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
fp, mp = calibration_curve(y, pr, n_bins=10, strategy='quantile')
ax[0].plot(mp, fp, 'o-'); ax[0].plot([0, .5], [0, .5], '--', color='grey')
ax[0].set(xlabel='predicted risk', ylabel='observed rate', title='Calibration')
q = pd.qcut(pr, 10, labels=False); lift = y.groupby(q).mean()
ax[1].bar(lift.index + 1, lift.values); ax[1].axhline(p, color='grey', ls='--')
ax[1].set(xlabel='predicted-risk decile (10 = highest)', ylabel='observed readmit rate', title='Observed rate by predicted risk')
plt.show()
top = pr >= np.quantile(pr, .9)
print(f'Top 10% flagged: readmit rate {y[top].mean():.3f} vs base {p:.3f} (lift {y[top].mean()/p:.2f}x); '
      f'captures {100*y[top].sum()/y.sum():.1f}% of readmissions')
lo, hi = boot(pr); print(f"{best}: AUC {res.loc[best, 'AUC']:.3f} (95% CI {lo:.3f}-{hi:.3f})")

## 5. Does a person's own history help?
Section 1 showed readmission is strongly clustered within people (observed between-person SD 0.27 vs 0.15 expected by chance). So we add features built **only from that person's earlier stays** (shifted so the current stay's own outcome is never included): number of earlier stays that were followed by a readmission, and the rate. Same grouped CV as before, so a person is never in both train and test.

In [ ]:
h = df.sort_values(['person_key', 'visit_number']).reset_index(drop=True)
assert (h.groupby('person_key').visit_number.diff().dropna() == 1).all()   # no gaps in visit sequence
gp = h.groupby('person_key')
h['prior_n'] = gp.cumcount()
h['prior_readmit_n'] = gp.readmit_30d.cumsum() - h.readmit_30d          # excludes the current stay
h['prior_readmit_rate'] = h.prior_readmit_n / h.prior_n.replace(0, np.nan)
print(h.groupby('prior_readmit_n').readmit_30d.agg(['mean', 'size']).head(8).rename(columns={'mean': 'readmit rate now'}))

yh = h.readmit_30d; base_cols = ['visit_number', 'birth_year', 'bmi', 'length_of_stay_days', 'n_medications', 'lab_a', 'lab_b', 'postcode_area']
hist_cols = base_cols + ['prior_readmit_n', 'prior_readmit_rate']
hf = list(StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED).split(h, yh, h.person_key))
out = []
for name, cols in [('Stay features only', base_cols), ('+ own readmission history', hist_cols)]:
    prh = np.zeros(len(h))
    for tr, te in hf:
        m = HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=200, l2_regularization=1.0, random_state=SEED)
        m.fit(h.iloc[tr][cols], yh.iloc[tr]); prh[te] = m.predict_proba(h.iloc[te][cols])[:, 1]
    hv = (h.prior_n > 0).values
    out.append(dict(model=name, AUC_all=roc_auc_score(yh, prh), AUC_first_stay=roc_auc_score(yh[~hv], prh[~hv]),
                    AUC_repeat_stays=roc_auc_score(yh[hv], prh[hv]), PR_AUC=average_precision_score(yh, prh)))
pd.DataFrame(out).set_index('model').round(3)

## 6. Conclusion

**Short answer: partly.**

* **From the stay itself** (length of stay, number of medications, lab A are the only features with visible signal; age, BMI, sex, lab B, postcode and visit number add essentially nothing), we get a cross-validated AUC of about **0.62–0.64**. That is real (the confidence interval excludes 0.5) but weak: flagging the top 10% of stays gives ~37% readmission vs 20% base, and catches under a fifth of all readmissions. Not enough to act on for individual decisions.
* **Adding the person's earlier outcomes** lifts AUC to about **0.82** (about 0.84 on repeat stays). Readmission is strongly a person-level tendency: 10% readmission for people with no earlier readmissions, rising to 50%+ for people with 3+.
* This does not help for a person's first stay (no history; AUC is ~0.60-0.62 there).

**Caveats**
* The history features assume the outcome of an earlier stay is known when predicting a later one. There are no dates in the data, so I cannot confirm that. If a prior stay's "readmitted" flag is only known because the next stay already happened, then at the moment of prediction for that next stay it is known, but you should check this against how the score would be used (e.g. at discharge vs at admission).
* Person-grouped CV was used throughout; a random row split would overstate performance for every model here.
* Only associations are shown; nothing here says what causes readmission. The data is one source with no external validation set, and only ~1,500 people.
* Models are untuned defaults; small gains from tuning are possible but the person-level structure, not model choice, is what drives the gap.